In [1]:
# Imports
from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from IPython.display import Image, display
from typing import Literal
import os

print("✅ All imports successful")

✅ All imports successful


In [2]:
# Load API key
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found! Please set it in your .env file.")

print("✅ API key loaded")

✅ API key loaded


In [3]:
# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    api_key=openai_api_key
)

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


In [4]:
# IMPORTANT: Replace this path with your PDF file
file_path = "NASS-Journal_Nigeria-Tax-Bill.pdf"

# Check if file exists
if not os.path.exists(file_path):
    print(f"⚠️ File not found: {file_path}")
    print("Please update the file_path variable with your PDF file.")
    print("\nFor this demo, we'll create sample documents instead...")
    
    # Create sample documents for demo
    from langchain_core.documents import Document
    pages = [
        Document(page_content="Nigeria Tax Bill, 2024", 
                metadata={"page": 1}),
        Document(page_content="Proteins are made of amino acids and perform many functions in cells.",
                metadata={"page": 2}),
        Document(page_content="DNA stores genetic information using four nucleotide bases.",
                metadata={"page": 3}),
    ]
    print("✅ Using sample documents for demo")
else:
    # Load the PDF
    loader = PyPDFLoader(file_path)
    pages = []
    
    # Load pages (async loading)
    async for page in loader.alazy_load():
        pages.append(page)
    
    print(f"✅ Loaded {len(pages)} pages from PDF")

✅ Loaded 213 pages from PDF


In [5]:
# Create text splitter (Module 2 knowledge!)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Characters per chunk
    chunk_overlap=100     # Overlap to preserve context
)

# Split documents
doc_splits = text_splitter.split_documents(pages)

print(f"✅ Created {len(doc_splits)} chunks")
print(f"\nSample chunk:")
print(f"{doc_splits[0].page_content[:200]}...")

✅ Created 624 chunks

Sample chunk:
NASS 
 NIGERIA TAX BILL, 2024 
 TABLE OF CONTENTS 
 CHAPTER ONE 
 OBJECTIVES AND APPLICATION 
 1.  Objective of the Act 
 2.  Application 
 CHAPTER TWO 
 TAXATION OF INCOME OF PERSONS 
 PART I 
 IMPOS...


In [8]:
embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    api_key = openai_api_key
)

print(" Embedding model initialized ♻")

 Embedding model initialized ♻


In [9]:
# Create Chroma vector store
chroma_path = "./chroma_db_agentic_rag"

# Create vector store from documents
vectorstore = Chroma(
    collection_name="agentic_rag_docs",
    persist_directory=chroma_path,
    embedding_function=embeddings
)

# Add documents
vectorstore.add_documents(documents=doc_splits)

print(f"✅ Vector store created with {len(doc_splits)} chunks")
print(f"   Persisted to: {chroma_path}")

✅ Vector store created with 624 chunks
   Persisted to: ./chroma_db_agentic_rag


In [ ]:
NIGERIA TAX BILL, 2024
A BILL FOR AN ACT TO REPEAL CERTAIN ACTS ON TAXATION AND
CONSOLIDATE THE LEGAL FRAMEWORKS RELATING TO TAXATION AND
ENACT THE NIGERIA TAX ACT TO PROVIDE FOR TAXATION OF INCOME,
TRANSACTIONS AND INSTRUMENTS, AND FOR RELATED MATTERS.